In [1]:
"""
Created on Tue Oct 22 20:48:49 2024

@author: yuri
email: syuri@tju.edu.cn
"""

import os

# Fix: avoid protobuf python-impl override that can crash TF on Kaggle.
os.environ.pop("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", None)
os.environ.pop("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", None)
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "cpp")

os.environ["KERAS_BACKEND"] = "tensorflow"

NEEDTRAIN = True  # train the model
LOAD_MODELS_FROM = "modelsxxxxxxx"  # the path of trained model weights for testing

_cwd_parts = os.getcwd().split(os.sep)
_root = _cwd_parts[1] if len(_cwd_parts) > 1 else ""
if _root in ("home", "content"):
    PLATFORM = "local"
    if os.path.exists("./input/"):
        for dir_name in os.listdir("./input/"):
            if dir_name[:6] == "models":
                LOAD_MODELS_FROM = dir_name
elif _root == "kaggle":
    PLATFORM = "kaggle"
    NEEDTRAIN = True
    for dir_name in os.listdir("/kaggle/input/"):
        if dir_name[:6] == "models":
            LOAD_MODELS_FROM = dir_name
else:
    if os.path.exists("/kaggle/input"):
        PLATFORM = "kaggle"
        NEEDTRAIN = True
        for dir_name in os.listdir("/kaggle/input/"):
            if dir_name[:6] == "models":
                LOAD_MODELS_FROM = dir_name
    else:
        PLATFORM = "local"

DATATYPE = ["eeg"]  # *** spe, eeg, stft, img ***
print(DATATYPE)

if PLATFORM == "local":
    LOAD_MODELS_FROM = f"./input/{LOAD_MODELS_FROM}"
    LOAD_DATA_FROM = "./input/hms-harmful-brain-activity-classification"
elif PLATFORM == "kaggle":
    LOAD_MODELS_FROM = f"/kaggle/input/{LOAD_MODELS_FROM}"
    LOAD_DATA_FROM = "/kaggle/input/hms-harmful-brain-activity-classification"
else:
    LOAD_DATA_FROM = "./input/hms-harmful-brain-activity-classification"

SFREQ = 200
RSFREQ = 200

EEG_LENGTH = 50
EEG_LENGTH_USED = 50
EEG_CHANNEL_USED = 16
EEG_MULTIPLY = 1

IMG_LENGTH = 20
IMG_HIGH = 324
IMG_WIDE = 324

SPE_HIGH = 100
SPE_WIDE = 256

STFT_LENGTH = 45
STFT_TIME = 0.15
STFT_HIGH = 32
STFT_WIDE = round(STFT_LENGTH / STFT_TIME)

filter_range = [0.5, 45]
filter_range2 = [0.1, 35]

SEED = 2024
BATCHSIZE = 16
LEARN_RATE = 1e-3
EPOCHS = 15
PATIENCE = 5
SPLITS = 5

READ_EEG_FILES = False
READ_SPE_FILES = False

spectrograms = {}
eegs = {}
stfts = {}
imgs = {}

spectrograms_test = {}
eegs_test = {}
stfts_test = {}
imgs_test = {}

BRAIN = [
    "Fp1-F7",
    "F7-T3",
    "T3-T5",
    "T5-O1",  # LL
    "Fp1-F3",
    "F3-C3",
    "C3-P3",
    "P3-O1",  # LP
    "Fz-Cz",
    "Cz-Pz",
    "Fp2-F4",
    "F4-C4",
    "C4-P4",
    "P4-O2",  # RP
    "Fp2-F8",
    "F8-T4",
    "T4-T6",
    "T6-O2",  # RL
]

TEST_BATCHSIZE = 128

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"

import warnings

warnings.filterwarnings("ignore")

import time
import gc
import io

USE_IMG = "img" in DATATYPE
USE_SPE = "spe" in DATATYPE
USE_STFT = "stft" in DATATYPE
if USE_IMG:
    from PIL import Image
    import matplotlib
    import matplotlib.pyplot as plt

import pandas as pd, numpy as np

if PLATFORM == "local":
    PREPROCESS_DIR = os.path.join(".", "input", "preprocess")
elif PLATFORM == "kaggle":
    PREPROCESS_DIR = os.path.join("/kaggle/working", "input", "preprocess")
else:
    PREPROCESS_DIR = os.path.join(".", "input", "preprocess")
os.makedirs(PREPROCESS_DIR, exist_ok=True)

_df0 = pd.read_csv(os.path.join(LOAD_DATA_FROM, "train.csv"))
TARGETS = _df0.columns[-6:]
print("Train shape:", _df0.shape)
print("Targets", list(TARGETS))


def _write_uniform_submission():
    ss_path = os.path.join(LOAD_DATA_FROM, "sample_submission.csv")
    if not os.path.exists(ss_path):
        ss_path = os.path.join(
            "/kaggle/input/hms-harmful-brain-activity-classification",
            "sample_submission.csv",
        )
    sub = pd.read_csv(ss_path)
    p = np.full((len(sub), len(TARGETS)), 1.0 / len(TARGETS), dtype=np.float64)
    sub[TARGETS] = p
    p2 = sub[TARGETS].values
    p2 = np.clip(p2, 1e-7, 1.0)
    p2 = p2 / p2.sum(axis=1, keepdims=True)
    sub[TARGETS] = p2.astype(np.float32)
    sub.to_csv("submission.csv", index=False)
    print("Wrote uniform baseline submission.csv", sub.shape)
    print(sub.head())


TF_AVAILABLE = True
try:
    import tensorflow as tf
    from tensorflow.keras import optimizers
    from tensorflow.keras.models import clone_model
except Exception as e:
    TF_AVAILABLE = False
    print("WARNING: TensorFlow import failed; will write uniform submission.")
    print("TF import error:", repr(e))

from scipy import signal

needed_leads = sorted({p for ch in BRAIN for p in ch.split("-")})
brain_pairs = [ch.split("-") for ch in BRAIN]

if filter_range is not None:
    b, a = signal.butter(3, np.float32(filter_range) * 2 / RSFREQ, "bandpass")

if not TF_AVAILABLE:
    _write_uniform_submission()
    raise SystemExit(0)

print(tf.version.VERSION)
print(tf.config.list_physical_devices("GPU"))

gpus = tf.config.list_physical_devices("GPU")
if len(gpus) <= 1:
    strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
    print(f"Using {len(gpus)} GPU")
else:
    strategy = tf.distribute.MirroredStrategy()
    print(f"Using {len(gpus)} GPUs")

np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

MIX = False
if MIX:
    policy = tf.keras.mixed_precision.Policy("mixed_float16")
    tf.keras.mixed_precision.set_global_policy(policy)
else:
    tf.keras.mixed_precision.set_global_policy("float32")
    print("Using full precision (faster end-to-end here due to fewer casts)")

df = _df0

# Fix: Do NOT early-exit with uniform submission on Kaggle when no weights are found.
# The previous code forced NEEDTRAIN=False on Kaggle without weights, causing poor score.
# We keep NEEDTRAIN=True so training/inference runs end-to-end.
if PLATFORM == "kaggle":
    if (not os.path.isdir(LOAD_MODELS_FROM)) or (
        len(os.listdir(LOAD_MODELS_FROM)) == 0
    ):
        print(
            f"INFO: No model folder found at {LOAD_MODELS_FROM}. Will train from scratch."
        )
    else:
        _maybe_weights = any(
            fn.endswith("_stage2.weights.h5") for fn in os.listdir(LOAD_MODELS_FROM)
        )
        if _maybe_weights:
            NEEDTRAIN = False
            print(
                f"INFO: Found weights in {LOAD_MODELS_FROM}; setting NEEDTRAIN=False for runtime."
            )

# Fix: Ensure `train` is always defined whenever NEEDTRAIN=True, so downstream cells don't NameError.
if NEEDTRAIN:
    TARGETS_RAW = [i + "_raw" for i in TARGETS]

    # Build a deduplicated training table with normalized targets (as in original logic).
    train = df.drop_duplicates(
        [
            "eeg_id",
            "seizure_vote",
            "lpd_vote",
            "gpd_vote",
            "lrda_vote",
            "grda_vote",
            "other_vote",
        ]
    ).reset_index(drop=True)
    train["sign_id"] = train.index.values

    y_data = train[TARGETS].values.astype(np.float32)
    train[TARGETS_RAW] = y_data
    y_norm = y_data / y_data.sum(axis=1, keepdims=True)
    train[TARGETS] = y_norm
    train.to_csv("train.csv", index=False)

    _key_cols = [
        "eeg_id",
        "seizure_vote",
        "lpd_vote",
        "gpd_vote",
        "lrda_vote",
        "grda_vote",
    ]
    _subcols = [
        "spectrogram_id",
        "spectrogram_label_offset_seconds",
        "eeg_label_offset_seconds",
        "eeg_sub_id",
    ]
    _df_min = df[_key_cols + _subcols].copy()
    GROUPS_LOOKUP = {
        k: g.reset_index(drop=True) for k, g in _df_min.groupby(_key_cols, sort=False)
    }

    _group_cache = {}
    for k, g in GROUPS_LOOKUP.items():
        eeg_sub_id = g["eeg_sub_id"].to_numpy(dtype=np.int64)
        r_spe = np.rint(
            g["spectrogram_label_offset_seconds"].to_numpy(dtype=np.float32) / 2.0
        ).astype(np.int32)
        r_eeg = g["eeg_label_offset_seconds"].to_numpy(dtype=np.float32)
        order = np.argsort(eeg_sub_id, kind="mergesort")
        ksel_valid = int(order[len(order) // 2]) if len(order) else 0
        _group_cache[k] = {
            "r_spe": r_spe,
            "r_eeg": r_eeg,
            "eeg_sub_id": eeg_sub_id,
            "ksel_valid": ksel_valid,
        }
    GROUPS_LOOKUP = None
    del _df_min
    gc.collect()



['eeg']
Train shape: (96950, 15)
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']


E0000 00:00:1778648458.637927      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778648458.641059      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TF import error: ImportError("cannot import name '_message' from 'google.protobuf.pyext' (/usr/local/lib/python3.11/dist-packages/google/protobuf/pyext/__init__.py)")
Wrote uniform baseline submission.csv (9850, 7)
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  2578018731      0.166667  0.166667  0.166667   0.166667   0.166667   
1  2578018731      0.166667  0.166667  0.166667   0.166667   0.166667   
2  2578018731      0.166667  0.166667  0.166667   0.166667   0.166667   
3  2578018731      0.166667  0.166667  0.166667   0.166667   0.166667   
4  1706196508      0.166667  0.166667  0.166667   0.166667   0.166667   

   other_vote  
0    0.166667  
1    0.166667  
2    0.166667  
3    0.166667  
4    0.166667  


SystemExit: 0

In [2]:
if NEEDTRAIN:
    PATH = os.path.join(LOAD_DATA_FROM, "train_eegs") + "/"

    if not READ_EEG_FILES:
        eeg_npy_path = os.path.join(PREPROCESS_DIR, "eegs.npy")
        if ("eeg" in DATATYPE) and (not os.path.exists(eeg_npy_path)):
            print(
                f"INFO: Missing {eeg_npy_path}. Switching READ_EEG_FILES=True to build eeg cache from parquet."
            )
            READ_EEG_FILES = True

    if ("eeg" in DATATYPE) and NEEDTRAIN and (not READ_EEG_FILES) and (len(eegs) == 0):
        if not os.path.exists(os.path.join(PREPROCESS_DIR, "eegs.npy")):
            READ_EEG_FILES = True

    if READ_EEG_FILES:
        if (("stft" in DATATYPE) or ("img" in DATATYPE)) and filter_range2 is not None:
            b2, a2 = signal.butter(
                3, np.float32(filter_range2) * 2 / RSFREQ, "bandpass"
            )
        time_start_time = time.time()

        lead_to_idx = {c: k for k, c in enumerate(needed_leads)}
        idx0 = np.fromiter((lead_to_idx[a0] for a0, _ in brain_pairs), dtype=np.int32)
        idx1 = np.fromiter((lead_to_idx[a1] for _, a1 in brain_pairs), dtype=np.int32)

        uniq_ids = train.eeg_id.unique()
        for i, eeg_id in enumerate(uniq_ids):
            if i % 200 == 0:
                gc.collect()
                xx = time.time() - time_start_time
                yy = xx / (i + 1) * len(uniq_ids)
                print(i, f"time: {round(xx / 60, 2)} min / {round(yy / 60, 2)} min")

            eeg_default = pd.read_parquet(
                os.path.join(PATH, (str(eeg_id) + ".parquet")),
                columns=needed_leads,
            )
            arr = eeg_default.to_numpy(dtype=np.float32, copy=False)

            eeg = arr[:, idx0].T - arr[:, idx1].T
            np.nan_to_num(eeg, copy=False, nan=0.0)

            if SFREQ != RSFREQ:
                eeg = signal.resample_poly(eeg, RSFREQ, SFREQ, axis=1)

            if "stft" in DATATYPE:
                eeg2 = signal.filtfilt(b, a, eeg, axis=1)
                nperseg = round(RSFREQ * 0.2)
                ff, tt, ss = signal.spectrogram(
                    eeg2,
                    axis=1,
                    fs=RSFREQ,
                    nperseg=nperseg,
                    noverlap=round(nperseg - RSFREQ * STFT_TIME),
                    nfft=320,
                )
                np.nan_to_num(ss, copy=False, nan=0.0)
                ss = ss[:, (ff > 0) * (ff <= 20), :]

            eeg = np.clip(eeg, a_min=-1024, a_max=1024)
            if filter_range is not None:
                eeg = signal.filtfilt(b, a, eeg, axis=1)
            eeg = np.array(eeg, dtype=np.float32)

            if "eeg" in DATATYPE:
                eegs[eeg_id] = eeg
            if "stft" in DATATYPE:
                stfts[eeg_id] = ss
                stfts[-eeg_id] = tt

        os.makedirs(PREPROCESS_DIR, exist_ok=True)
        if "eeg" in DATATYPE:
            np.save(os.path.join(PREPROCESS_DIR, "eegs.npy"), eegs, allow_pickle=True)
        if "stft" in DATATYPE:
            np.save(os.path.join(PREPROCESS_DIR, "stfts.npy"), stfts, allow_pickle=True)
        if "img" in DATATYPE:
            np.save(os.path.join(PREPROCESS_DIR, "imgs.npy"), imgs, allow_pickle=True)

    else:
        if "eeg" in DATATYPE:
            eegs = np.load(
                os.path.join(PREPROCESS_DIR, "eegs.npy"), allow_pickle=True
            ).item()
        if "stft" in DATATYPE:
            stfts = np.load(
                os.path.join(PREPROCESS_DIR, "stfts.npy"), allow_pickle=True
            ).item()
        if "img" in DATATYPE:
            imgs = np.load(
                os.path.join(PREPROCESS_DIR, "imgs.npy"), allow_pickle=True
            ).item()



INFO: Missing /kaggle/working/input/preprocess/eegs.npy. Switching READ_EEG_FILES=True to build eeg cache from parquet.


NameError: name 'train' is not defined

In [3]:
if NEEDTRAIN:
    PATH = os.path.join(LOAD_DATA_FROM, "train_spectrograms") + "/"
    files = os.listdir(PATH)
    print(f"There are {len(files)} spectrogram parquets")
    time_start_time = time.time()
    if READ_SPE_FILES:
        for i, f in enumerate(files):
            if i % 200 == 0:
                gc.collect()
                xx = time.time() - time_start_time
                yy = xx / (i + 1) * len(files)
                print(i, f"time: {round(xx / 60, 2)} min / {round(yy / 60, 2)} min")
            tmp = pd.read_parquet(f"{PATH}{f}")
            name = int(f.split(".")[0])
            spectrograms[name] = tmp.iloc[:, 1:].values
        os.makedirs(PREPROCESS_DIR, exist_ok=True)
        np.save(
            os.path.join(PREPROCESS_DIR, "spectrograms.npy"),
            spectrograms,
            allow_pickle=True,
        )
    else:
        if "spe" in DATATYPE:
            spectrograms = np.load(
                os.path.join(PREPROCESS_DIR, "spectrograms.npy"), allow_pickle=True
            ).item()




There are 10024 spectrogram parquets


In [4]:
class DataGenerator(tf.keras.utils.Sequence):
    def __init__(
        self,
        dataframe,
        batch_size=32,
        shuffle=False,
        sample_weights=False,
        mode="train",
        eegs=None,
        stfts=None,
        specs=None,
        imgs=None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.sample_weights = sample_weights
        self.mode = mode
        self.eegs = eegs
        self.stfts = stfts
        self.specs = specs
        self.imgs = imgs

        self._eeg_id = self.dataframe["eeg_id"].to_numpy()
        self._sign_id = (
            self.dataframe["sign_id"].to_numpy()
            if "sign_id" in self.dataframe.columns
            else np.arange(len(self.dataframe), dtype=np.int64)
        )
        if self.mode != "test":
            self._targets = self.dataframe[list(TARGETS)].to_numpy(dtype=np.float32)
            self._targets_raw = self.dataframe[list(TARGETS_RAW)].to_numpy(
                dtype=np.float32
            )
            self._votes_key = self._targets_raw[:, :5]  # seizure,lpd,gpd,lrda,grda
        else:
            self._targets = None
            self._targets_raw = None
            self._votes_key = None

        if "spe" in DATATYPE:
            self._spectrogram_id = self.dataframe["spectrogram_id"].to_numpy()

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataframe) / self.batch_size))

    def __getitem__(self, index):
        indexes = self.indexes[index * self.batch_size : (index + 1) * self.batch_size]
        x, y, sample_weights = self.__data_generation(indexes)
        return x, y, sample_weights

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.dataframe))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, indexes):
        if "spe" in DATATYPE:
            x_spe = np.zeros((len(indexes), 4, SPE_HIGH, SPE_WIDE), dtype="float32")
        if "eeg" in DATATYPE:
            x_eeg = np.zeros(
                (
                    len(indexes),
                    EEG_CHANNEL_USED * EEG_MULTIPLY,
                    round(EEG_LENGTH_USED * RSFREQ / EEG_MULTIPLY),
                ),
                dtype="float32",
            )
        if "img" in DATATYPE:
            x_img = np.zeros((len(indexes), IMG_HIGH, IMG_WIDE, 3), dtype="float32")

        y = np.zeros((len(indexes), len(TARGETS)), dtype="float32")
        sample_weights = np.zeros((len(indexes), 1), dtype="float32")

        for j, i in enumerate(indexes):
            eeg_id = int(self._eeg_id[i])
            sign_id = int(self._sign_id[i])

            if self.mode != "test":
                sample_weight = float(self._targets_raw[i].sum() / 20.0)

            if self.mode == "test":
                r_spe = 0
                r_eeg = 0.0
            else:
                key = (
                    eeg_id,
                    float(self._votes_key[i, 0]),
                    float(self._votes_key[i, 1]),
                    float(self._votes_key[i, 2]),
                    float(self._votes_key[i, 3]),
                    float(self._votes_key[i, 4]),
                )
                g = _group_cache[key]
                if self.mode == "train":
                    ksel = np.random.randint(0, len(g["r_spe"]))
                elif self.mode == "valid":
                    ksel = int(g["ksel_valid"])
                else:
                    ksel = 0
                r_spe = int(g["r_spe"][ksel])
                r_eeg = float(g["r_eeg"][ksel])

            if "spe" in DATATYPE:
                spe_id = int(self._spectrogram_id[i])
                spe = []
                base = self.specs[spe_id]
                for k in range(4):
                    spe.append(
                        np.reshape(
                            base[r_spe : (r_spe + 300), k * 100 : (k + 1) * 100].T,
                            (1, 100, 300),
                        )
                    )
                spe = np.concatenate(spe, axis=0)

            if "eeg" in DATATYPE:
                eeg = self.eegs[eeg_id][
                    :, round(r_eeg * RSFREQ) : round((r_eeg + 50) * RSFREQ)
                ]

            if "img" in DATATYPE:
                img = self.imgs[sign_id]

            if "spe" in DATATYPE:
                spe[np.isnan(spe)] = 0
                exp_min, exp_max = -4, 6
                spe = np.clip(spe, a_min=np.exp(exp_min), a_max=np.exp(exp_max))
                spe = np.log(spe)
                spe = spe[
                    :,
                    :,
                    round((spe.shape[2] - SPE_WIDE) / 2) : -round(
                        (spe.shape[2] - SPE_WIDE) / 2
                    ),
                ]

                if self.mode == "train":
                    spe2 = spe.copy()
                    if np.random.rand() > 0.5:
                        spe[0] = spe2[2]
                        spe[2] = spe2[0]
                    if np.random.rand() > 0.5:
                        spe[1] = spe2[3]
                        spe[3] = spe2[1]
                    if np.random.rand() > 0.5:
                        spe[0] = spe2[1]
                        spe[2] = spe2[3]
                        spe[1] = spe2[0]
                        spe[3] = spe[2]

                spe = (spe - exp_min) / (exp_max - exp_min) * 255
                spe = np.clip(spe, a_min=0, a_max=255)
                x_spe[j] = spe

            if "eeg" in DATATYPE:
                eeg = eeg[
                    :,
                    round((EEG_LENGTH - EEG_LENGTH_USED) * RSFREQ / 2) : round(
                        (EEG_LENGTH + EEG_LENGTH_USED) * RSFREQ / 2
                    ),
                ]
                eeg_save = np.zeros((x_eeg.shape[1], x_eeg.shape[2]), dtype=np.float32)

                eeg = np.concatenate(
                    (
                        eeg[0 : round(EEG_CHANNEL_USED / 2), :],
                        eeg[-round(EEG_CHANNEL_USED / 2) :, :],
                    ),
                    axis=0,
                )

                if self.mode == "train":
                    if np.random.rand() > 0.5:
                        mask = round(np.random.rand() * eeg.shape[1])
                        eeg[
                            :,
                            mask : round(mask + np.random.rand() * eeg.shape[1] * 0.02),
                        ] = 0
                    if np.random.rand() > 0.5:
                        mask = round(np.random.rand() * eeg.shape[1])
                        eeg[
                            :,
                            mask : round(mask + np.random.rand() * eeg.shape[1] * 0.02),
                        ] = 0
                    if np.random.rand() > 0.5:
                        mask = round(np.random.rand() * eeg.shape[1])
                        eeg[
                            :,
                            mask : round(mask + np.random.rand() * eeg.shape[1] * 0.02),
                        ] = 0

                    if np.random.rand() > 0.5:
                        eeg[np.random.permutation(eeg.shape[0])[0], :] = 0
                    if np.random.rand() > 0.5:
                        eeg[np.random.permutation(eeg.shape[0])[0], :] = 0

                    eeg[0 : round(EEG_CHANNEL_USED / 2), :] = eeg[
                        0 : round(EEG_CHANNEL_USED / 2), :
                    ][np.random.permutation(8), :]
                    eeg[-round(EEG_CHANNEL_USED / 2) :, :] = eeg[
                        -round(EEG_CHANNEL_USED / 2) :, :
                    ][np.random.permutation(8), :]

                    eeg2 = eeg.copy()
                    eeg[4:8, :] = eeg2[12:16, :]
                    eeg[8:12, :] = eeg2[4:8, :]
                    eeg[12:16, :] = eeg2[8:12, :]

                    if np.random.rand() > 0.5:
                        eeg = eeg[::-1, :]

                    for ii in range(eeg_save.shape[0]):
                        eeg_save[ii, :] = eeg[
                            ii // EEG_MULTIPLY, ii % EEG_MULTIPLY :: EEG_MULTIPLY
                        ]
                else:
                    eeg2 = eeg.copy()
                    eeg[4:8, :] = eeg2[12:16, :]
                    eeg[8:12, :] = eeg2[4:8, :]
                    eeg[12:16, :] = eeg2[8:12, :]
                    for ii in range(eeg_save.shape[0]):
                        eeg_save[ii, :] = eeg[
                            ii // EEG_MULTIPLY, ii % EEG_MULTIPLY :: EEG_MULTIPLY
                        ]

                eeg = np.clip(eeg_save, a_min=-255, a_max=255)
                eeg = eeg + 255
                eeg = eeg / 2
                x_eeg[j] = eeg

            if "img" in DATATYPE:
                img_save = np.zeros((IMG_HIGH, IMG_WIDE), dtype=np.float32)
                if self.mode == "train":
                    img[0:8, :, :] = img[0:8, :, :][np.random.permutation(8), :, :]
                    img[10:18, :, :] = img[10:18, :, :][np.random.permutation(8), :, :]
                    if np.random.rand() > 0.5:
                        img = img[::-1, :, :]

                for ii in range(img.shape[0]):
                    axis_temp = img_save.shape[1] / img.shape[0] / 2 * (2 * ii + 1)
                    start_temp = round(
                        max(axis_temp - img_save.shape[1] / img.shape[0], 0)
                    )
                    end_temp = round(
                        min(
                            img_save.shape[0],
                            axis_temp + img_save.shape[1] / img.shape[0],
                        )
                    )
                    temp_temp = round(img.shape[1] / 2 - (axis_temp - start_temp))
                    img_save[start_temp:end_temp, :] = (
                        img_save[start_temp:end_temp, :]
                        + img[
                            ii, temp_temp : round(temp_temp + end_temp - start_temp), :
                        ]
                    )
                img_save = np.clip(img_save, a_min=0, a_max=1)
                img = np.reshape(img_save, (img_save.shape[0], img_save.shape[1], 1))
                img = np.concatenate((img, img, img), -1)
                img = (img - np.mean(img)) / (np.std(img) + 1e-6)
                x_img[j] = img

            if self.mode != "test":
                yy = self._targets[i]
                y[j] = yy / float(yy.sum())
                sample_weights[j] = sample_weight if self.sample_weights else 1.0
            else:
                sample_weights[j] = 1.0

        x = {}
        if "spe" in DATATYPE:
            x["spe"] = x_spe
        if "eeg" in DATATYPE:
            x["eeg"] = x_eeg
        if "img" in DATATYPE:
            x["img"] = x_img

        return x, y, sample_weights




NameError: name 'tf' is not defined

In [5]:
class CosineAnnealingLRScheduler(optimizers.schedules.LearningRateSchedule):
    def __init__(self, total_step, lr_max, lr_min=0, warmth_rate=0):
        super(CosineAnnealingLRScheduler, self).__init__()
        self.total_step = total_step
        if warmth_rate == 0:
            self.warm_step = 1
        else:
            self.warm_step = int(warmth_rate)
        self.lr_max = lr_max
        self.lr_min = lr_min

    def __call__(self, step):
        step = step + 1
        if step < self.warm_step:
            lr = self.lr_max / self.warm_step * step
        else:
            lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (
                1.0
                + tf.cos(
                    (step - self.warm_step) / (self.total_step - self.warm_step) * np.pi
                )
            )
        return np.float32(lr)


class IniToOne(tf.keras.initializers.Initializer):
    def __init__(self):
        super(IniToOne, self).__init__()

    def __call__(self, shape, dtype=None):
        assert len(shape) == 3
        filter_length, input_channel, filter_count = shape
        kernel = np.zeros(shape, dtype=np.float32)
        for i in range(filter_count):
            kernel[i % filter_length, 0, i] = 1.0
        kernel = tf.convert_to_tensor(kernel, dtype=dtype)
        return kernel

    def get_config(self):
        return {}


class SumToOne(tf.keras.constraints.Constraint):
    def __init__(self):
        super(SumToOne, self).__init__()

    def __call__(self, w):
        w = tf.abs(w)
        w_normed = w / tf.reduce_sum(w, axis=[0, 1], keepdims=True)
        return w_normed

    def get_config(self):
        return {}


class IniToOneAtten(tf.keras.initializers.Initializer):
    def __init__(self):
        super(IniToOneAtten, self).__init__()

    def __call__(self, shape, dtype=None):
        assert len(shape) == 3
        filter_length, input_channel, filter_count = shape
        kernel = np.zeros(shape, dtype=np.float32)
        kernel[(filter_length - 1) // 2 : (filter_length) // 2 + 1, :, :] = 1 / (
            (filter_length) // 2 + 1 - (filter_length - 1) // 2
        )
        kernel = tf.convert_to_tensor(kernel, dtype=dtype)
        return kernel

    def get_config(self):
        return {}


class SumToOneAtten(tf.keras.constraints.Constraint):
    def __init__(self):
        super(SumToOneAtten, self).__init__()

    def __call__(self, w):
        w = tf.abs(w)
        w_normed = w / tf.reduce_sum(w, axis=[0, 1], keepdims=True)
        return w_normed

    def get_config(self):
        return {}


class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, feat_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.ffn = tf.keras.Sequential(
            [
                tf.keras.layers.Dense(ff_dim, activation="gelu"),
                tf.keras.layers.Dense(feat_dim),
            ]
        )
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training):
        attn_output, weights = self.att(inputs, inputs, return_attention_scores=True)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output), weights


class ClassToken(tf.keras.layers.Layer):
    """Append a class token to an input layer."""

    def build(self, input_shape):
        cls_init = tf.zeros_initializer()
        self.hidden_size = input_shape[-1]
        self.cls = tf.Variable(
            name="cls",
            initial_value=cls_init(shape=(1, 1, self.hidden_size), dtype="float32"),
            trainable=True,
        )

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        cls_broadcasted = tf.cast(
            tf.broadcast_to(self.cls, [batch_size, 1, self.hidden_size]),
            dtype=inputs.dtype,
        )
        return tf.concat([cls_broadcasted, inputs], 1)




NameError: name 'optimizers' is not defined

In [6]:
def build_model():
    inp = list()

    if "eeg" in DATATYPE:
        inp_eeg = tf.keras.Input(
            shape=(
                EEG_CHANNEL_USED * EEG_MULTIPLY,
                round(EEG_LENGTH_USED * RSFREQ / EEG_MULTIPLY),
            ),
            name="eeg",
        )
        x_eeg_raw = tf.keras.layers.Reshape((inp_eeg.shape[1], inp_eeg.shape[2], 1))(
            inp_eeg
        )

        strides = 10
        if PLATFORM == "local":
            eeg_embed = tf.keras.layers.Conv1D(
                filters=strides * 3,
                kernel_size=strides,
                strides=strides,
                padding="same",
                use_bias=False,
                activation=None,
                kernel_initializer=IniToOne(),
                kernel_constraint=SumToOne(),
                input_shape=(None, 1),
            )
        else:
            eeg_embed = tf.keras.layers.Conv1D(
                filters=strides * 3,
                kernel_size=strides,
                strides=strides,
                padding="same",
                use_bias=False,
                activation=None,
            )

        x_eeg = tf.keras.layers.TimeDistributed(eeg_embed)(x_eeg_raw)

        x_eeg = tf.keras.layers.Concatenate(axis=-1)(
            [
                tf.keras.layers.Reshape((x_eeg.shape[1], x_eeg.shape[2], -1, 1))(
                    x_eeg[:, :, :, 0 * strides : 1 * strides]
                ),
                tf.keras.layers.Reshape((x_eeg.shape[1], x_eeg.shape[2], -1, 1))(
                    x_eeg[:, :, :, 1 * strides : 2 * strides]
                ),
                tf.keras.layers.Reshape((x_eeg.shape[1], x_eeg.shape[2], -1, 1))(
                    x_eeg[:, :, :, 2 * strides : 3 * strides]
                ),
            ]
        )
        x_eeg = tf.keras.layers.Permute([4, 2, 1, 3])(x_eeg)
        x_eeg = tf.keras.layers.Reshape((x_eeg.shape[1], x_eeg.shape[2], -1))(x_eeg)
        x_eeg = tf.keras.layers.Permute((3, 2, 1))(x_eeg)

        base_model_eeg = tf.keras.applications.EfficientNetV2B0(
            include_top=False, weights=None, include_preprocessing=True
        )
        if NEEDTRAIN:
            if PLATFORM == "local":
                wt = f"./input/pre-trained-weights/{base_model_eeg.name}_notop.h5"
                if os.path.exists(wt):
                    base_model_eeg.load_weights(wt)
            if PLATFORM == "kaggle":
                wt = f"/kaggle/input/pre-trained-weights/{base_model_eeg.name}_notop.h5"
                if os.path.exists(wt):
                    base_model_eeg.load_weights(wt)
        base_model_eeg._name = "eeg_extractor"

        x_eeg = base_model_eeg(x_eeg)
        x_eeg = x_eeg[:, :, (x_eeg.shape[2] - 1) // 2 : (x_eeg.shape[2]) // 2 + 1, :]
        x_eeg = tf.keras.layers.GlobalAveragePooling2D()(x_eeg)
        x_eeg = tf.keras.layers.Dropout(0.5)(x_eeg)

        inp.append(inp_eeg)
        y_eeg = tf.keras.layers.Dense(
            len(TARGETS), activation="softmax", dtype="float32"
        )(x_eeg)

    y = y_eeg * 1
    model = tf.keras.Model(inputs=inp, outputs=y)
    return model




In [7]:
weights_ok = False
if (not NEEDTRAIN) and os.path.isdir(LOAD_MODELS_FROM):
    needed = [
        os.path.join(LOAD_MODELS_FROM, f"fold{fi}_stage2.weights.h5")
        for fi in range(SPLITS)
    ]
    weights_ok = all(os.path.exists(p) for p in needed)



In [8]:
if NEEDTRAIN and (not weights_ok):
    os.makedirs("models", exist_ok=True)
    SAVE_MODELS_TO = "models"

    if "sign_id" not in train.columns:
        train["sign_id"] = np.arange(len(train), dtype=np.int64)

    patients = train["patient_id"].to_numpy()
    uniq_pat = np.unique(patients)
    rng = np.random.RandomState(SEED)
    rng.shuffle(uniq_pat)
    pat_folds = np.array_split(uniq_pat, SPLITS)

    with strategy.scope():
        model_template = build_model()

    for fold_i in range(SPLITS):
        valid_pats = set(pat_folds[fold_i].tolist())
        is_valid = np.array([p in valid_pats for p in patients])
        tr_df = train.loc[~is_valid].reset_index(drop=True)
        va_df = train.loc[is_valid].reset_index(drop=True)

        print(f"\nFold {fold_i}: train={len(tr_df)} valid={len(va_df)}")

        with strategy.scope():
            model = clone_model(model_template)
            model.compile(
                optimizer=tf.keras.optimizers.Adam(learning_rate=LEARN_RATE),
                loss=tf.keras.losses.KLDivergence(reduction="sum_over_batch_size"),
            )

        train_gen = DataGenerator(
            tr_df,
            batch_size=BATCHSIZE,
            shuffle=True,
            sample_weights=True,
            mode="train",
            eegs=eegs,
            stfts=stfts,
            specs=spectrograms,
            imgs=imgs,
        )
        valid_gen = DataGenerator(
            va_df,
            batch_size=BATCHSIZE,
            shuffle=False,
            sample_weights=True,
            mode="valid",
            eegs=eegs,
            stfts=stfts,
            specs=spectrograms,
            imgs=imgs,
        )

        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=os.path.join(
                    SAVE_MODELS_TO, f"fold{fold_i}_stage2.weights.h5"
                ),
                monitor="val_loss",
                save_best_only=True,
                save_weights_only=True,
                mode="min",
                verbose=1,
            ),
        ]

        model.fit(
            train_gen,
            validation_data=valid_gen,
            epochs=EPOCHS,
            callbacks=callbacks,
            verbose=2,
        )

        tf.keras.backend.clear_session()
        gc.collect()

    LOAD_MODELS_FROM = SAVE_MODELS_TO
    weights_ok = True
    NEEDTRAIN = False



NameError: name 'train' is not defined

In [9]:
if (not NEEDTRAIN) and weights_ok:
    TARGETS_RAW = [t + "_raw" for t in TARGETS]

    models = list()
    model_template = build_model()

    for model_i in range(SPLITS):
        print(f"Fold {model_i + 1}")
        model = clone_model(model_template)
        model.load_weights(
            os.path.join(LOAD_MODELS_FROM, f"fold{model_i}_stage2.weights.h5")
        )
        models.append(model)

    ss_path = os.path.join(LOAD_DATA_FROM, "sample_submission.csv")
    if not os.path.exists(ss_path):
        ss_path = os.path.join(
            "/kaggle/input/hms-harmful-brain-activity-classification",
            "sample_submission.csv",
        )
    sub = pd.read_csv(ss_path)

    test = pd.read_csv(os.path.join(LOAD_DATA_FROM, "test.csv"))

    test = test.set_index("eeg_id").reindex(sub["eeg_id"].values).reset_index()
    test["sign_id"] = np.arange(len(test), dtype=np.int64)
    print("Test shape", test.shape, "Submission template shape", sub.shape)

    PATH_test = os.path.join(LOAD_DATA_FROM, "test_eegs") + "/"

    if (
        ("spe" in DATATYPE)
        or ("eeg" in DATATYPE)
        or ("stft" in DATATYPE)
        or ("img" in DATATYPE)
    ):
        if filter_range2 is not None and ("img" in DATATYPE):
            b2, a2 = signal.butter(
                3, np.float32(filter_range2) * 2 / RSFREQ, "bandpass"
            )

        preds_all = np.zeros((len(test), len(TARGETS)), dtype=np.float32)

        lead_to_idx = {c: k for k, c in enumerate(needed_leads)}
        idx0 = np.fromiter((lead_to_idx[a0] for a0, _ in brain_pairs), dtype=np.int32)
        idx1 = np.fromiter((lead_to_idx[a1] for _, a1 in brain_pairs), dtype=np.int32)

        test_eeg_ids = test.eeg_id.to_numpy(dtype=np.int64)

        eeg_batch = np.empty(
            (
                TEST_BATCHSIZE,
                EEG_CHANNEL_USED * EEG_MULTIPLY,
                round(EEG_LENGTH_USED * RSFREQ / EEG_MULTIPLY),
            ),
            dtype=np.float32,
        )

        start_idx = 0
        for i, eeg_id in enumerate(test_eeg_ids):
            if i % 200 == 0:
                print(i, ", ", end="")

            eeg_default = pd.read_parquet(
                os.path.join(PATH_test, (str(int(eeg_id)) + ".parquet")),
                columns=needed_leads,
            )
            arr = eeg_default.to_numpy(dtype=np.float32, copy=False)

            eeg = arr[:, idx0].T - arr[:, idx1].T
            np.nan_to_num(eeg, copy=False, nan=0.0)

            if SFREQ != RSFREQ:
                eeg = signal.resample_poly(eeg, RSFREQ, SFREQ, axis=1)

            eeg = np.clip(eeg, a_min=-1024, a_max=1024)
            eegshape = eeg.shape[1]
            eeg = np.concatenate((eeg[:, ::-1], eeg, eeg[:, ::-1]), axis=1)
            if filter_range is not None:
                eeg = signal.filtfilt(b, a, eeg, axis=1)
            eeg = eeg[:, eegshape : eegshape * 2]
            eeg = np.array(eeg, dtype=np.float32)

            eeg = eeg[
                :,
                round((EEG_LENGTH - EEG_LENGTH_USED) * RSFREQ / 2) : round(
                    (EEG_LENGTH + EEG_LENGTH_USED) * RSFREQ / 2
                ),
            ]
            eeg = np.concatenate(
                (
                    eeg[0 : round(EEG_CHANNEL_USED / 2), :],
                    eeg[-round(EEG_CHANNEL_USED / 2) :, :],
                ),
                axis=0,
            )
            eeg2 = eeg.copy()
            eeg[4:8, :] = eeg2[12:16, :]
            eeg[8:12, :] = eeg2[4:8, :]
            eeg[12:16, :] = eeg2[8:12, :]

            eeg = np.clip(eeg, a_min=-255, a_max=255)
            eeg = (eeg + 255) / 2.0

            bi = i - start_idx
            eeg_batch[bi] = eeg

            if ((i + 1) % TEST_BATCHSIZE == 0) or ((i + 1) == len(test_eeg_ids)):
                end_idx = i + 1
                bs = end_idx - start_idx
                x_batch = {"eeg": eeg_batch[:bs]}

                preds = None
                for m in models:
                    p = m(x_batch, training=False).numpy().astype(np.float32)
                    preds = p if preds is None else (preds + p)
                pred = (preds / float(SPLITS)).astype(np.float32)

                preds_all[start_idx:end_idx, :] = pred

                gc.collect()
                start_idx = end_idx

    sub[TARGETS] = preds_all

    p = sub[TARGETS].values.astype(np.float64)
    p = np.clip(p, 1e-7, 1.0)
    p = p / p.sum(axis=1, keepdims=True)
    sub[TARGETS] = p.astype(np.float32)

    sub.to_csv("submission.csv", index=False)
    print("Submission shape", sub.shape)
    print(sub.head())
else:
    if not os.path.exists("submission.csv"):
        _write_uniform_submission()